# ELECTRA + ScalarMix — LLM-judge clean dataset (from Drive)

Train on **Shrishti / Option E** `clean_dataset` splits from Google Drive:
- `train.csv`, `val.csv`, `test.csv` with label column **`education_level_judge`**

Evaluate OOD (also from Drive, judge labels only):
- `ood_onestop.csv`, `ood_race-middle.csv`, `ood_race-high.csv`

**No DANN.** No Hugging Face download for eval.

Saves model, metrics JSON, summary CSV, and confusion matrices to `DRIVE_OUT_DIR`.

In [ ]:
!pip install -q transformers scikit-learn torch pandas matplotlib seaborn tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── Paths (edit to your Drive folder with clean_dataset CSVs) ────────────
DRIVE_CLEAN_DIR = "/content/drive/MyDrive/beyond_flesch/clean_dataset"
DRIVE_OUT_DIR = "/content/drive/MyDrive/beyond_flesch/trail/electra_llm_judge"

MODEL_NAME = "google/electra-large-discriminator"
TEXT_COL = "full_text"
LABEL_COL = "education_level_judge"

MAX_LEN = 512
BATCH_SIZE = 4
EPOCHS = 3
LR = 2e-5
WARMUP_STEPS = 100
RNG_SEED = 42
LABEL_SMOOTHING = 0.1
BALANCE_TRAIN = False

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}
EVAL_LABELS = [0, 1, 2]

In [ ]:
from __future__ import annotations

import json
import os
import random
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.nn import Parameter, ParameterList
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

CLEAN_DIR = Path(DRIVE_CLEAN_DIR)
if not CLEAN_DIR.joinpath("train.csv").exists():
    raise FileNotFoundError(
        f"Missing train.csv under {CLEAN_DIR}. "
        "Upload llm_as_a_judge/outputs/clean_dataset/*.csv to Drive and set DRIVE_CLEAN_DIR."
    )

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
CM_DIR = os.path.join(DRIVE_OUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)
print("Device:", device)
print("CLEAN_DIR:", CLEAN_DIR)
print("OUT:", DRIVE_OUT_DIR)

In [ ]:
# ── ScalarMix + ELECTRA classifier (no DANN) ─────────────────────────────
class ScalarMix(nn.Module):
    def __init__(self, mixture_size: int, trainable: bool = True) -> None:
        super().__init__()
        self.scalar_parameters = ParameterList(
            [Parameter(torch.zeros(1), requires_grad=trainable) for _ in range(mixture_size)]
        )
        self.gamma = Parameter(torch.ones(1), requires_grad=trainable)

    def forward(self, tensors: List[torch.Tensor]) -> torch.Tensor:
        w = torch.nn.functional.softmax(
            torch.cat([p for p in self.scalar_parameters]), dim=0
        )
        w = torch.split(w, 1)
        return self.gamma * sum(weight * t for weight, t in zip(w, tensors))


class ElectraScalarMixClassifier(nn.Module):
    def __init__(self, model_name: str, num_classes: int = 3, dropout: float = 0.2) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = int(self.encoder.config.hidden_size)
        n_layers = int(self.encoder.config.num_hidden_layers) + 1
        self.scalar_mix = ScalarMix(n_layers)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def encode_pooled(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        mixed = self.dropout(self.scalar_mix(list(out.hidden_states)))
        mask = attention_mask.unsqueeze(-1).float()
        summed = (mixed * mask).sum(dim=1)
        return summed / mask.sum(dim=1).clamp(min=1e-9)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        return self.head(self.encode_pooled(input_ids, attention_mask))

print("Model OK")

In [ ]:
# ── Load CSVs from Drive ───────────────────────────────────────────────────
def load_split_csv(name: str) -> pd.DataFrame:
    path = CLEAN_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
    df[TEXT_COL] = df[TEXT_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
    bad = ~df[LABEL_COL].isin(label2id)
    if bad.any():
        print(f"  [{name}] dropping {bad.sum()} rows with unknown labels")
        df = df[~bad].reset_index(drop=True)
    df["label_id"] = df[LABEL_COL].map(label2id).astype(int)
    return df


df_train = load_split_csv("train")
df_val = load_split_csv("val")
df_test = load_split_csv("test")

ood_splits = {
    "ood_onestop": load_split_csv("ood_onestop"),
    "ood_race-middle": load_split_csv("ood_race-middle"),
    "ood_race-high": load_split_csv("ood_race-high"),
}

if BALANCE_TRAIN:
    min_n = df_train[LABEL_COL].value_counts().min()
    print(f"Balancing train to {min_n} per judge class")
    parts = []
    for lab in label2id:
        sub = df_train[df_train[LABEL_COL] == lab]
        parts.append(sub.sample(n=min(min_n, len(sub)), random_state=RNG_SEED))
    df_train = pd.concat(parts).sample(frac=1, random_state=RNG_SEED).reset_index(drop=True)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print("Train judge labels:\n", df_train[LABEL_COL].value_counts())
for oname, odf in ood_splits.items():
    print(f"{oname}: n={len(odf)}  judge={dict(odf[LABEL_COL].value_counts())}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextClsDataset(Dataset):
    def __init__(self, texts: List[str], labels: np.ndarray) -> None:
        self.texts = texts
        self.labels = labels.astype(int)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def df_to_loader(df: pd.DataFrame, shuffle: bool) -> DataLoader:
    return DataLoader(
        TextClsDataset(df[TEXT_COL].tolist(), df["label_id"].values),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
    )

train_loader = df_to_loader(df_train, shuffle=True)
val_loader = df_to_loader(df_val, shuffle=False)
test_loader = df_to_loader(df_test, shuffle=False)

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────
model = ElectraScalarMixClassifier(MODEL_NAME).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = EPOCHS * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps
)

history = []
best_val_f1 = -1.0
best_path = os.path.join(DRIVE_OUT_DIR, "best_model.pt")


@torch.no_grad()
def eval_loader(loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys, preds = [], []
    for batch in loader:
        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
        )
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        ys.extend(batch["label"].numpy().tolist())
    return np.array(ys), np.array(preds)


for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        optimizer.zero_grad()
        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
        )
        loss = criterion(logits, batch["label"].to(device))
        loss.backward()
        optimizer.step()
        scheduler.step()
        running += loss.item()
    y_v, p_v = eval_loader(val_loader)
    val_f1 = f1_score(y_v, p_v, labels=EVAL_LABELS, average="macro", zero_division=0)
    val_acc = accuracy_score(y_v, p_v)
    avg_loss = running / max(len(train_loader), 1)
    print(f"Epoch {epoch+1}: loss={avg_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_f1:.4f}")
    history.append({"epoch": epoch + 1, "train_loss": avg_loss, "val_acc": val_acc, "val_macro_f1": val_f1})
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_path)
        print(f"  → saved best checkpoint (val macro-F1 {val_f1:.4f})")

model.load_state_dict(torch.load(best_path, map_location=device))
print("Training done. Best val macro-F1:", best_val_f1)

In [ ]:
# ── Eval: in-distribution test + OOD (judge labels) + confusion matrices ─

def save_confusion_matrix(y_true, y_pred, title, out_path, labels):
    names = [id2label[i] for i in labels]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True (judge)")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


@torch.no_grad()
def predict_loader(loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    return eval_loader(loader)


@torch.no_grad()
def predict_texts(texts: List[str], batch_size: int = 16) -> np.ndarray:
    model.eval()
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="predict", leave=False):
        enc = tokenizer(
            texts[i : i + batch_size],
            truncation=True,
            max_length=MAX_LEN,
            padding=True,
            return_tensors="pt",
        )
        logits = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.array(preds, dtype=int)


def eval_split(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> Dict:
    labels = EVAL_LABELS
    present = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    acc = accuracy_score(y_true, y_pred)
    f1_all = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    f1_present = f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)
    report = classification_report(
        y_true, y_pred, labels=labels,
        target_names=[id2label[i] for i in labels], zero_division=0,
    )
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(report)
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro-F1 (3-class): {f1_all:.4f}  Macro-F1 (present classes {present}): {f1_present:.4f}")
    cm_path = os.path.join(CM_DIR, f"cm_{name.replace(' ', '_').lower()}.png")
    save_confusion_matrix(y_true, y_pred, name, cm_path, labels)
    print(f"Saved CM → {cm_path}")
    return {
        "corpus": name,
        "n": int(len(y_true)),
        "accuracy": float(acc),
        "macro_f1_3class": float(f1_all),
        "macro_f1_present": float(f1_present),
        "gold_classes": present,
        "classification_report": report,
        "confusion_matrix_png": cm_path,
    }


results = {
    "config": {
        "model": MODEL_NAME,
        "label_col": LABEL_COL,
        "clean_dir": str(CLEAN_DIR),
        "dann": False,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "best_val_macro_f1": float(best_val_f1),
        "train_history": history,
    },
    "evaluations": [],
}

y_t, p_t = predict_loader(test_loader)
results["evaluations"].append(eval_split("in_domain_test", y_t, p_t))

for oname, odf in ood_splits.items():
    y_true = odf["label_id"].values
    y_pred = predict_texts(odf[TEXT_COL].tolist())
    results["evaluations"].append(eval_split(oname, y_true, y_pred))

summary = pd.DataFrame([
    {
        "corpus": e["corpus"],
        "n": e["n"],
        "accuracy": e["accuracy"],
        "macro_f1_3class": e["macro_f1_3class"],
        "macro_f1_present": e["macro_f1_present"],
    }
    for e in results["evaluations"]
])
print("\nSummary:\n", summary.to_string(index=False))

json_path = os.path.join(DRIVE_OUT_DIR, "eval_results.json")
csv_path = os.path.join(DRIVE_OUT_DIR, "eval_summary.csv")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
summary.to_csv(csv_path, index=False)
print(f"\nSaved JSON → {json_path}")
print(f"Saved CSV  → {csv_path}")
print(f"Model      → {best_path}")